# CS313 Nhóm 5 – E3a: chạy lại DeepSVDD và ALOCC (mỗi epoch, seed 0/1/2)

Mục đích: kiểm chứng vì sao ALOCC và DeepSVDD trong bàn giao E3a thấp hơn bài báo (xem `docs/E3a_ban_giao.md`).
Mỗi epoch đều chấm test và validation, nên `summarize.py` cho cả `best_test` và `last`.

| Model | Biến thể | Ghi chú |
|---|---|---|
| DeepSVDD | `full` | đúng code gốc MTGFlow: BatchNorm ở eval mode từ epoch 2 |
| DeepSVDD | `bn_train_each_epoch` | cách bàn giao E3a (train mode mỗi epoch) |
| ALOCC | `full` | điểm −D(x) như code MTGFlow |
| ALOCC | `score_DRx` | điểm −D(R(x)) như bài báo ALOCC gốc, cùng lần train |

**Trước khi chạy:**
1. *Settings → Accelerator*: **GPU T4**.
2. *Settings → Internet*: **On** (để clone repo).
3. *Add Input*: bộ dữ liệu PSM/MSL/SMD (file `damr_data.zip` của nhóm, hoặc bộ TSLib). Notebook tự tìm file theo tên.
4. Nên bấm **Save Version → Save & Run All (Commit)** để chạy nền, không cần mở tab (khoảng 30–60 phút).

In [ ]:
REPO = "https://github.com/Waterbear566/cs313-damr-reproduce.git"
BRANCH = "e3a-hoang-khang"   # runner DeepSVDD/ALOCC chỉ có ở nhánh này; đổi thành "main" sau khi merge E3a
!rm -rf /kaggle/working/repo && git clone -q -b $BRANCH $REPO /kaggle/working/repo
%cd /kaggle/working/repo
!git log --oneline -1

In [ ]:
import platform, torch, numpy, sklearn, pandas
print("Python ", platform.python_version(), "| PyTorch", torch.__version__, "| CUDA", torch.version.cuda)
print("numpy  ", numpy.__version__, "| pandas", pandas.__version__, "| sklearn", sklearn.__version__)
print("GPU    ", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "KHÔNG CÓ GPU – bật GPU T4 trong Settings")
assert torch.cuda.is_available()

## 1. Dữ liệu: tìm file trong /kaggle/input và nối vào `DAMR/data/`

In [ ]:
import glob, os
NEED = {"psm": ["train.csv", "test.csv", "test_label.csv"],
        "msl": ["MSL_train.npy", "MSL_test.npy", "MSL_test_label.npy"],
        "smd": ["SMD_train.npy", "SMD_test.npy", "SMD_test_label.npy"]}
for ds, files in NEED.items():
    cands = [d for d in {os.path.dirname(p) for p in glob.glob(f"/kaggle/input/**/{files[0]}", recursive=True)}
             if all(os.path.exists(os.path.join(d, f)) for f in files)]
    if ds == "psm":
        cands = sorted(cands, key=lambda d: "psm" not in d.lower())
    assert cands, f"{ds.upper()}: chưa thấy trong /kaggle/input"
    os.makedirs(f"DAMR/data/{ds}", exist_ok=True)
    for f in files:
        dst = f"DAMR/data/{ds}/{f}"
        if not os.path.exists(dst): os.symlink(os.path.join(cands[0], f), dst)
    print(f"{ds.upper()}: dùng {cands[0]}")
!python scripts/check_data.py --root DAMR/data --datasets PSM MSL SMD

## 2. Chạy thử 2 epoch trên PSM (ghi vào `results/smoke_*.csv`, không lẫn vào kết quả thật)

In [ ]:
!python baselines/DeepSVDD/run.py --dataset PSM --seed 0 --epochs 2 --exp smoke
!python baselines/ALOCC/run.py --dataset PSM --seed 0 --epochs 2 --exp smoke

## 3. Chạy thật: 3 dataset × 3 seed (DeepSVDD 2 biến thể + ALOCC)

In [ ]:
import time
t0 = time.time()
!bash scripts/run_e3a_rerun.sh
print(f"Tổng thời gian: {(time.time() - t0) / 60:.1f} phút")

## 4. Tổng hợp: mean ± std 3 seed, `best_test` và `last`, cạnh số bài báo

In [ ]:
!python scripts/summarize.py results/E3_DeepSVDD.csv results/E3_ALOCC.csv --out results/summary_E3a_rerun.csv

## 5. Đóng gói kết quả để tải về

Tải `/kaggle/working/e3a_rerun_results.zip` (tab **Output** của bản đã Save, hoặc panel bên phải).
Trong zip: `results/E3_DeepSVDD.csv`, `results/E3_ALOCC.csv` (đã gồm cả dòng cũ của Hoàng Khang), bảng tổng hợp và log.
Điểm từng epoch (`checkpoints/E3/**/scores/`) để riêng trong `e3a_rerun_scores.zip`, dùng cho E5 và để tính lại F1.

In [ ]:
!zip -q -r /kaggle/working/e3a_rerun_results.zip results/E3_DeepSVDD.csv results/E3_ALOCC.csv results/summary_E3a_rerun.csv logs/E3_*.log
!zip -q -r /kaggle/working/e3a_rerun_scores.zip checkpoints/E3 -x "*.pt"
!ls -lh /kaggle/working/*.zip